In [5]:
# Environment + LLM setup
import os
from dotenv import load_dotenv

load_dotenv()

from langchain_groq import ChatGroq

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

llm = ChatGroq(model="openai/gpt-oss-120b")


c:\Users\harsu\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
result = llm.invoke("Hello")
result

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'reasoning_content': 'The user just says "Hello". We need to respond appropriately. Probably greet back. No special constraints. Provide friendly response.'}, response_metadata={'token_usage': {'completion_tokens': 44, 'prompt_tokens': 72, 'total_tokens': 116, 'completion_time': 0.094024191, 'prompt_time': 0.003575121, 'queue_time': 0.035131498, 'total_time': 0.097599312, 'completion_tokens_details': {'reasoning_tokens': 26}}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_1d982b31b2', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019bb3a8-df9a-7742-9f65-7b58e311cd2d-0', usage_metadata={'input_tokens': 72, 'output_tokens': 44, 'total_tokens': 116, 'output_token_details': {'reasoning': 26}})

In [8]:
# Imports for structured planning + graph state
from typing import Annotated, List
import operator
from typing_extensions import TypedDict
from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage, SystemMessage

In [9]:
# Blog outline + blog meta schemas 

class Section(BaseModel):
    name: str = Field(description="Name for this section of the blog post")
    description: str = Field(description="Brief overview of what this blog section will cover")

class Sections(BaseModel):
    sections: List[Section] = Field(description="Sections of the blog post")

class BlogMeta(BaseModel):
    title: str = Field(description="A compelling, clear blog post title")
    tldr: str = Field(description="A short TL;DR summary (2-4 sentences)")


In [10]:
# Planners 
outline_planner = llm.with_structured_output(Sections)
meta_planner = llm.with_structured_output(BlogMeta)


In [11]:
# LangGraph constants + State definitions
from langgraph.constants import Send

class State(TypedDict):
    topic: str
    title: str
    tldr: str
    sections: list[Section]
    completed_sections: Annotated[list, operator.add]
    final_blog: str

class WorkerState(TypedDict):
    section: Section
    completed_sections: Annotated[list, operator.add]


C:\Users\harsu\AppData\Local\Temp\ipykernel_17904\3073068911.py:2: LangGraphDeprecatedSinceV10: Importing Send from langgraph.constants is deprecated. Please use 'from langgraph.types import Send' instead. Deprecated in LangGraph V1.0 to be removed in V2.0.
  from langgraph.constants import Send


In [ ]:
# Nodes (orchestrator + workers + synthesizer)

def orchestrator(state: State):
    """Orchestrator: generates blog title, TL;DR, and an outline."""

    # 1) Generate title + TL;DR
    meta = meta_planner.invoke(
        [
            SystemMessage(content="You are a technical blog editor. Produce a strong title and a concise TL;DR."),
            HumanMessage(content=f"Blog topic: {state['topic']}"),
        ]
    )

    # 2) Generate outline (sections)
    outline = outline_planner.invoke(
        [
            SystemMessage(
                content=(
                    "Create a detailed outline for a technical blog post. "
                    "Return 5–8 sections. Each section must be practical and logically ordered."
                )
            ),
            HumanMessage(content=f"Blog topic: {state['topic']}"),
        ]
    )

    print("Blog Meta:", meta)
    print("Blog Sections:", outline)

    return {
        "title": meta.title,
        "tldr": meta.tldr,
        "sections": outline.sections,
    }


def llm_call(state: WorkerState):
    """Worker: writes a single blog section."""

    section = llm.invoke(
        [
            SystemMessage(
                content=(
                    "Write a blog section for a technical tutorial. "
                    "No preamble. Use markdown. Be concrete and instructional. "
                    "Assume the reader is a developer. Do not add SEO metadata."
                )
            ),
            HumanMessage(
                content=(
                    f"Section name: {state['section'].name}\n"
                    f"Section description: {state['section'].description}\n\n"
                    "Write this section now."
                )
            ),
        ]
    )

    return {"completed_sections": [section.content]}


def assign_workers(state: State):
    """Create one worker per section."""
    return [Send("llm_call", {"section": s}) for s in state["sections"]]


def synthesizer(state: State):
    """Combine title + TL;DR + all sections into a final blog markdown doc."""

    completed_sections = state["completed_sections"]
    body = "\n\n---\n\n".join(completed_sections)

    final_blog = f"# {state['title']}\n\n**TL;DR:** {state['tldr']}\n\n---\n\n{body}"
    return {"final_blog": final_blog}
